In [10]:
import torch
import torch.nn as nn
from torchvision import models
from torch.nn.functional import relu
import torchinfo


import os
import numpy as np
import cv2
from glob import glob
from tqdm import tqdm
from skimage.io import imread, imsave
from sklearn.model_selection import train_test_split

In [8]:
IMAGE_DIR = "data/images/"
MASK_DIR = "data/annotations/masks/"
OUTPUT_DIR = "data/preprocessed/"
LABEL_MAP = {
    "unannotated": 0,
    "other": 1,
    "non-invasive epithelium": 2,
    "invasive epithelium": 3,
    "necrosis": 4
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
class UNet(nn.Module):
    def __init__(self, n_class=4):
        super().__init__()
        
        # Encoding / downsampling
        self.enc1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # Decoding / upsampling
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)

        # Final output layer
        self.out_conv = nn.Conv2d(32, n_class, kernel_size=1)


    def forward(self, x):
        # Encoding/downsampling
        e1 = relu(self.enc1(x))          # 256x256x32

        e2 = relu(self.enc2(e1))         # 128x128x64

        e3 = relu(self.enc3(e2))         # 64x64x128

        # Decoding/upsampling
        d1 = relu(self.up1(e3))          # 128x128x64
        d2 = relu(self.up2(d1 + e2))     # 256x256x32

        out = self.out_conv(d2 + e1)       # 256x256x1

        return out


In [23]:
model = UNet()
torchinfo.summary(model, input_size=(3, 256, 256))

Layer (type:depth-idx)                   Output Shape              Param #
UNet                                     [4, 256, 256]             --
├─Conv2d: 1-1                            [32, 256, 256]            896
├─Conv2d: 1-2                            [64, 128, 128]            18,496
├─Conv2d: 1-3                            [128, 64, 64]             73,856
├─ConvTranspose2d: 1-4                   [64, 128, 128]            73,792
├─ConvTranspose2d: 1-5                   [32, 256, 256]            18,464
├─Conv2d: 1-6                            [4, 256, 256]             132
Total params: 185,636
Trainable params: 185,636
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.52
Input size (MB): 0.79
Forward/backward pass size (MB): 56.62
Params size (MB): 0.74
Estimated Total Size (MB): 58.15